# LeetCode 1249 — Minimum Remove to Make Valid Parentheses bằng Python

Link đề: https://leetcode.com/problems/minimum-remove-to-make-valid-parentheses/ · Độ khó: **Medium** · Chủ đề: String, Stack

Skeleton LeetCode:

```python
class Solution:
    def minRemoveToMakeValid(self, s: str) -> str:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** ("return any valid string"), nên khi kiểm thử ta dùng **hàm kiểm tra hợp lệ**: kết quả phải là chuỗi ngoặc hợp lệ, là dãy con của `s`, giữ nguyên mọi chữ cái, và xoá **ít ngoặc nhất**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Kiểm tra một chuỗi ngoặc có hợp lệ bằng **bộ đếm cân bằng** (balance counter) và hiểu vì sao chỉ cần một biến đếm khi có một loại ngoặc.
2. Dùng **stack lưu chỉ số** để biết chính xác ngoặc nào không được ghép cặp.
3. Viết lời giải **hai lượt quét** (trái→phải xoá `)` thừa, phải→trái xoá `(` thừa) không cần stack.
4. Chứng minh số ngoặc bị xoá là **tối thiểu**: bằng số `)` không ghép được + số `(` không ghép được.
5. Viết **validator** để test bài nhiều đáp án, và tránh các lỗi: xoá nhầm `(` bên trái, dùng `str.replace`, quên xử lý `(` thừa.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm `'('`, `')'` và chữ cái thường tiếng Anh. Hãy xoá **ít ngoặc nhất** (`'('` hoặc `')'`, ở bất kỳ vị trí nào) để chuỗi còn lại là chuỗi ngoặc **hợp lệ**, và trả về **một** chuỗi hợp lệ bất kỳ như vậy.

Định nghĩa hợp lệ:

- Chuỗi rỗng hoặc chỉ gồm chữ cái thường là hợp lệ;
- Nếu `A`, `B` hợp lệ thì `AB` hợp lệ;
- Nếu `A` hợp lệ thì `(A)` hợp lệ.

**Ví dụ 1:** `s = "lee(t(c)o)de)"` → `"lee(t(c)o)de"` (cũng chấp nhận `"lee(t(co)de)"`, `"lee(t(c)ode)"`).

**Ví dụ 2:** `s = "a)b(c)d"` → `"ab(c)d"`.

**Ví dụ 3:** `s = "))(("` → `""` (chuỗi rỗng cũng hợp lệ).

**Ràng buộc:** `1 <= s.length <= 10⁵`; `s[i]` là `'('`, `')'` hoặc chữ cái thường.

In [1]:
import itertools
import random
import time
from collections import deque
from typing import List

examples = [("lee(t(c)o)de)", "lee(t(c)o)de"), ("a)b(c)d", "ab(c)d"), ("))((", "")]
for s, sample in examples:
    print(repr(s), "->", repr(sample))

'lee(t(c)o)de)' -> 'lee(t(c)o)de'
'a)b(c)d' -> 'ab(c)d'
'))((' -> ''


## 3. Kiến thức nền

### 3.1 Kiểm tra chuỗi ngoặc hợp lệ bằng biến đếm

Với **một loại ngoặc**, duyệt trái → phải, `bal += 1` khi gặp `(`, `bal -= 1` khi gặp `)`. Chuỗi hợp lệ ⇔ `bal` **không bao giờ âm** và **bằng 0** ở cuối.

In [2]:
def is_valid(t):
    bal = 0
    for c in t:
        if c == "(":
            bal += 1
        elif c == ")":
            bal -= 1
            if bal < 0:
                return False
    return bal == 0


for t in ["", "abc", "(a)(b)", "((c))", ")(", "(()", "lee(t(c)o)de)"]:
    print(repr(t), is_valid(t))

'' True
'abc' True
'(a)(b)' True
'((c))' True
')(' False
'(()' False
'lee(t(c)o)de)' False


### 3.2 Stack (ngăn xếp) với `list`

`list.append` để đẩy vào, `list.pop()` lấy phần tử cuối (LIFO — vào sau ra trước). Ở bài này ta đẩy **chỉ số** của `(`; gặp `)` thì `pop` để ghép với `(` **gần nhất** chưa ghép.

In [3]:
s = "a(b(c)d"
st = []
for i, c in enumerate(s):
    if c == "(":
        st.append(i)
    elif c == ")" and st:
        print(f"ghép ')' tại {i} với '(' tại {st.pop()}")
print("'(' chưa ghép tại:", st)

ghép ')' tại 5 với '(' tại 3
'(' chưa ghép tại: [1]


### 3.3 Dựng chuỗi hiệu quả: list + `''.join`

Chuỗi Python bất biến, nên xây kết quả bằng cách gom ký tự vào `list` rồi `''.join(...)` một lần (`O(n)`). Để "xoá" ký tự tại các chỉ số trong tập `bad`, dùng comprehension có điều kiện.

In [4]:
s = "lee(t(c)o)de)"
bad = {12}
print("".join(c for i, c in enumerate(s) if i not in bad))
chars = list(s)
chars[12] = ""
print("".join(chars))

lee(t(c)o)de
lee(t(c)o)de


### 3.4 Dãy con (subsequence) và kiểm tra bằng iterator

Kết quả phải là **dãy con** của `s` (chỉ xoá, không thêm/đổi thứ tự). Mẹo kiểm tra: `it = iter(s); all(c in it for c in t)` — mỗi `c in it` tiêu thụ iterator tới khi gặp `c`.

In [5]:
def is_subsequence(t, s):
    it = iter(s)
    return all(c in it for c in t)


print(is_subsequence("ab(c)d", "a)b(c)d"), is_subsequence("ba", "ab"))

True False


### 3.5 Validator cho bài nhiều đáp án

Số ngoặc tối thiểu phải xoá = số `)` không ghép được (khi `bal` sắp âm) + số `(` còn dư cuối cùng. Kết quả đúng ⇔ hợp lệ, là dãy con, giữ đủ chữ cái và xoá đúng số tối thiểu đó.

In [6]:
def min_removals(s):
    bal = extra_close = 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal == 0:
                extra_close += 1
            else:
                bal -= 1
    return extra_close + bal


def is_correct(s, res):
    letters = [c for c in s if c.isalpha()]
    return (is_valid(res) and is_subsequence(res, s)
            and [c for c in res if c.isalpha()] == letters
            and len(s) - len(res) == min_removals(s))


for s, sample in examples:
    print(repr(s), "min removals =", min_removals(s), "| mẫu đúng?", is_correct(s, sample))
print(is_correct("lee(t(c)o)de)", "lee(t(co)de)"), is_correct("lee(t(c)o)de)", "leet(c)ode"))

'lee(t(c)o)de)' min removals = 1 | mẫu đúng? True
'a)b(c)d' min removals = 1 | mẫu đúng? True
'))((' min removals = 4 | mẫu đúng? True
True False


## 4. Ý tưởng nền tảng / trực giác

Tính tay `s = "a)b(c)d"`, theo dõi `bal`:

| i | ký tự | bal trước | hành động | bal sau |
|---|---|---|---|---|
| 0 | a | 0 | giữ | 0 |
| 1 | ) | 0 | `bal` sẽ âm → **xoá** | 0 |
| 2 | b | 0 | giữ | 0 |
| 3 | ( | 0 | giữ | 1 |
| 4 | c | 1 | giữ | 1 |
| 5 | ) | 1 | ghép → giữ | 0 |
| 6 | d | 0 | giữ | 0 |

Kết quả `"ab(c)d"`. Còn với `"lee(t(c)o)de)"`: hai `(` ghép với hai `)` đầu; `)` cuối gặp lúc `bal = 0` → xoá.

Quan sát hai loại "lỗi": `)` xuất hiện khi không còn `(` nào mở (phát hiện ngay khi quét trái→phải), và `(` không bao giờ được đóng (chỉ biết khi quét hết — tức là các `(` còn lại trong stack / dư ở cuối).

In [7]:
s = "lee(t(c)o)de)"
bal = 0
for i, c in enumerate(s):
    if c == "(":
        bal += 1
    elif c == ")":
        if bal == 0:
            print(f"i={i}: ')' thừa -> xoá")
            continue
        bal -= 1
print("số '(' dư cuối cùng:", bal)

i=12: ')' thừa -> xoá
số '(' dư cuối cùng: 0


## 5. Cách 1 — Brute Force / baseline

Xoá dần theo **số lượng ngoặc bị xoá tăng dần** (`0, 1, 2, …`): thử mọi tổ hợp vị trí ngoặc cần xoá với kích thước đó, trả về chuỗi hợp lệ đầu tiên. Vì duyệt kích thước tăng dần nên kết quả xoá ít nhất.

In [8]:
class SolutionBrute:
    def minRemoveToMakeValid(self, s):
        idx = [i for i, c in enumerate(s) if c in "()"]
        for r in range(len(idx) + 1):
            for rem in itertools.combinations(idx, r):
                rem = set(rem)
                t = "".join(c for i, c in enumerate(s) if i not in rem)
                if is_valid(t):
                    return t
        return ""


for s, _ in examples:
    res = SolutionBrute().minRemoveToMakeValid(s)
    print(repr(s), "->", repr(res), is_correct(s, res))

'lee(t(c)o)de)' -> 'lee(t(co)de)' True
'a)b(c)d' -> 'ab(c)d' True
'))((' -> '' True


**Độ phức tạp:** gọi `p` là số ngoặc; worst case thử `2ᵖ` tập con, mỗi lần `O(n)` → `O(2ᵖ · n)`. Chỉ chạy được khi `p ≤ ~15`.

**Nút thắt:** ta không tận dụng cấu trúc ghép cặp: ngoặc nào "thừa" có thể biết **trực tiếp** khi quét.

## 6. Bước đột phá

**Insight:** ghép mỗi `)` với `(` **gần nhất bên trái** chưa ghép (đúng như stack). Những ngoặc **không ghép được** chính là những ngoặc phải xoá, và xoá đúng chúng là đủ + tối thiểu.

- `)` không ghép được ⇔ lúc gặp nó, không có `(` nào đang mở (`bal == 0` / stack rỗng).
- `(` không ghép được ⇔ còn lại trong stack khi quét xong.

**Invariant khi quét trái → phải:** stack chứa chỉ số các `(` **đã gặp nhưng chưa ghép**, theo thứ tự tăng dần; phần đã xử lý (bỏ những `)` thừa) có `bal ≥ 0`.

**Quy tắc:**

- Nếu gặp `(` → đẩy chỉ số vào stack.
- Nếu gặp `)` và stack khác rỗng → pop (ghép).
- Nếu gặp `)` và stack rỗng → đánh dấu xoá.
- Cuối cùng: mọi chỉ số còn trong stack → đánh dấu xoá.

**Đối xứng:** quét phải → trái thì vai trò `(` và `)` đổi nhau. Đây là cơ sở của lời giải hai lượt không cần stack.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Stack chỉ số + tập cần xoá

In [9]:
class SolutionStack:
    def minRemoveToMakeValid(self, s):
        st = []
        remove = set()
        for i, c in enumerate(s):
            if c == "(":
                st.append(i)
            elif c == ")":
                if st:
                    st.pop()
                else:
                    remove.add(i)
        remove.update(st)
        return "".join(c for i, c in enumerate(s) if i not in remove)


for s, _ in examples:
    res = SolutionStack().minRemoveToMakeValid(s)
    print(repr(s), "->", repr(res), is_correct(s, res))

'lee(t(c)o)de)' -> 'lee(t(c)o)de' True
'a)b(c)d' -> 'ab(c)d' True
'))((' -> '' True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (stack + set).

### 7.2 Cách 3 — Hai lượt quét với bộ đếm

- Lượt 1 (trái → phải): giữ ký tự, bỏ `)` khi `bal == 0`.
- Lượt 2 (phải → trái) trên kết quả lượt 1: giữ ký tự, bỏ `(` khi `bal == 0` (lúc này `bal` đếm `)` đang chờ).

In [10]:
class SolutionTwoPass:
    def minRemoveToMakeValid(self, s):
        first = []
        bal = 0
        for c in s:
            if c == "(":
                bal += 1
            elif c == ")":
                if bal == 0:
                    continue
                bal -= 1
            first.append(c)
        out = []
        bal = 0
        for c in reversed(first):
            if c == ")":
                bal += 1
            elif c == "(":
                if bal == 0:
                    continue
                bal -= 1
            out.append(c)
        return "".join(reversed(out))


for s, _ in examples:
    res = SolutionTwoPass().minRemoveToMakeValid(s)
    print(repr(s), "->", repr(res), is_correct(s, res))

'lee(t(c)o)de)' -> 'lee(t(c)o)de' True
'a)b(c)d' -> 'ab(c)d' True
'))((' -> '' True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho kết quả trung gian. Không cần stack/set.

### 7.3 Cách 4 — Một lượt + xoá `(` thừa **từ cuối**

Sau lượt 1 (như Cách 3), biến `bal` cho biết còn **bao nhiêu** `(` thừa. Ta xoá đúng `bal` dấu `(` **cuối cùng** (phải nhất) trong kết quả lượt 1. Vì sao lại là các `(` cuối? Một `(` thừa không bao giờ cần ghép, và xoá các `(` bên phải nhất đảm bảo các `(` còn lại đều có `)` đóng phía sau (xem mục 10).

In [11]:
class SolutionOnePassTrim:
    def minRemoveToMakeValid(self, s):
        first = []
        bal = 0
        for c in s:
            if c == "(":
                bal += 1
            elif c == ")":
                if bal == 0:
                    continue
                bal -= 1
            first.append(c)
        i = len(first) - 1
        while bal > 0:                  # xoá bal dấu '(' phải nhất
            if first[i] == "(":
                first[i] = ""
                bal -= 1
            i -= 1
        return "".join(first)


for s, _ in examples:
    res = SolutionOnePassTrim().minRemoveToMakeValid(s)
    print(repr(s), "->", repr(res), is_correct(s, res))

'lee(t(c)o)de)' -> 'lee(t(c)o)de' True
'a)b(c)d' -> 'ab(c)d' True
'))((' -> '' True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.4 Cách 5 — Stack trên list ký tự, sửa tại chỗ

Biến `s` thành `list`; khi phát hiện ngoặc thừa thì gán ký tự đó thành `''`. Gọn, một vòng lặp + một vòng dọn stack.

In [12]:
class SolutionInPlace:
    def minRemoveToMakeValid(self, s):
        chars = list(s)
        st = []
        for i, c in enumerate(chars):
            if c == "(":
                st.append(i)
            elif c == ")":
                if st:
                    st.pop()
                else:
                    chars[i] = ""
        for i in st:
            chars[i] = ""
        return "".join(chars)


for s, _ in examples:
    res = SolutionInPlace().minRemoveToMakeValid(s)
    print(repr(s), "->", repr(res), is_correct(s, res))

'lee(t(c)o)de)' -> 'lee(t(c)o)de' True
'a)b(c)d' -> 'ab(c)d' True
'))((' -> '' True


Các cách 2, 4, 5 cho **cùng** kết quả (cùng xoá các `(` không ghép được, chính là các `(` "phải nhất" theo nghĩa ghép cặp). Cách 3 cũng vậy. Brute force có thể cho đáp án **khác** nhưng vẫn đúng.

## 8. TRACE từng bước

Trace Cách 5 trên `s = "lee(t(c)o)de)"`:

| i | c | stack trước | hành động | stack sau |
|---|---|---|---|---|
| 3 | ( | [] | push 3 | [3] |
| 5 | ( | [3] | push 5 | [3, 5] |
| 7 | ) | [3, 5] | pop 5 (ghép 5–7) | [3] |
| 9 | ) | [3] | pop 3 (ghép 3–9) | [] |
| 12 | ) | [] | stack rỗng → xoá 12 | [] |

Cuối: stack rỗng → không có `(` thừa. Kết quả `"lee(t(c)o)de"`.

In [13]:
def trace_stack(s):
    chars = list(s)
    st = []
    for i, c in enumerate(chars):
        if c == "(":
            st.append(i)
            print(f"i={i:2} '(' -> push       | stack={st}")
        elif c == ")":
            if st:
                j = st.pop()
                print(f"i={i:2} ')' -> ghép với {j:2} | stack={st}")
            else:
                chars[i] = ""
                print(f"i={i:2} ')' -> thừa, xoá  | stack={st}")
    for i in st:
        chars[i] = ""
        print(f"cuối: '(' tại {i} không được đóng -> xoá")
    res = "".join(chars)
    print("kết quả:", repr(res))
    return res


trace_stack("lee(t(c)o)de)")

i= 3 '(' -> push       | stack=[3]
i= 5 '(' -> push       | stack=[3, 5]
i= 7 ')' -> ghép với  5 | stack=[3]
i= 9 ')' -> ghép với  3 | stack=[]
i=12 ')' -> thừa, xoá  | stack=[]
kết quả: 'lee(t(c)o)de'


'lee(t(c)o)de'

Trace thêm một chuỗi có cả `)` thừa lẫn `(` thừa:

In [14]:
trace_stack("))a(b(c)d((")

i= 0 ')' -> thừa, xoá  | stack=[]
i= 1 ')' -> thừa, xoá  | stack=[]
i= 3 '(' -> push       | stack=[3]
i= 5 '(' -> push       | stack=[3, 5]
i= 7 ')' -> ghép với  5 | stack=[3]
i= 9 '(' -> push       | stack=[3, 9]
i=10 '(' -> push       | stack=[3, 9, 10]
cuối: '(' tại 3 không được đóng -> xoá
cuối: '(' tại 9 không được đóng -> xoá
cuối: '(' tại 10 không được đóng -> xoá
kết quả: 'ab(c)d'


'ab(c)d'

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minRemoveToMakeValid(self, s: str) -> str:
        chars = list(s)
        st = []
        for i, c in enumerate(chars):
            if c == "(":
                st.append(i)
            elif c == ")":
                if st:
                    st.pop()
                else:
                    chars[i] = ""
        for i in st:
            chars[i] = ""
        return "".join(chars)


for s, _ in examples:
    assert is_correct(s, Solution().minRemoveToMakeValid(s))
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

**Kết quả hợp lệ.** Sau khi xoá, mỗi `)` còn lại đã được ghép với một `(` đứng **trước** nó, và mỗi `(` còn lại đã được ghép với một `)` đứng **sau** nó; các cặp ghép theo kiểu stack lồng nhau đúng cách (không cắt chéo). Do đó khi quét chuỗi mới, `bal` = số `(` đã mở chưa đóng ≥ 0 tại mọi thời điểm và về 0 ở cuối.

**Số xoá tối thiểu.** Gọi `C` = số `)` bị xoá ở lượt quét (khi `bal == 0`) và `O` = số `(` dư cuối cùng.

- *Cận dưới cho `)`:* xét tiền tố kết thúc ngay tại `)` thứ `t` bị xoá. Trong tiền tố này số `)` vượt số `(` đúng `t` đơn vị (mỗi lần xoá xảy ra khi đã "hết `(`"). Chuỗi hợp lệ yêu cầu mọi tiền tố có `#( ≥ #)`, nên phải xoá **ít nhất** `t` dấu `)` trong tiền tố đó. Với `t = C` → xoá ít nhất `C` dấu `)`.
- *Cận dưới cho `(`:* đối xứng (xét hậu tố, quét phải → trái) → xoá ít nhất `O` dấu `(`. Hai loại xoá khác nhau nên tổng ≥ `C + O`.
- Thuật toán xoá đúng `C + O` → tối ưu. ∎

**Vì sao Cách 4 xoá `(` phải nhất là đúng:** sau lượt 1, chuỗi `first` có mọi tiền tố với `bal(j) ≥ 0` và `bal` cuối bằng `O`. Gọi `r(j)` = số `(` bị xoá ở vị trí `≤ j`; `bal` mới tại `j` là `bal(j) − r(j)`. Nếu `r(j) = 0` thì hiển nhiên `≥ 0`. Nếu `r(j) > 0` thì (vì xoá các `(` **phải nhất**) mọi `(` sau `j` cũng bị xoá; phần sau `j` chứa `O − r(j)` dấu `(` và `c` dấu `)`, nên `O = bal(j) + (O − r(j)) − c`, suy ra `bal(j) − r(j) = c ≥ 0`. `bal` mới ở cuối là `O − O = 0`. Vậy kết quả hợp lệ. ∎

## 11. Time / Space Complexity

Đặt `n = len(s)`, `p` = số ngoặc.

| Cách | Time | Space |
|---|---|---|
| 1. Brute (thử tập con) | `O(2ᵖ · n)` | `O(n)` |
| 2. Stack + set | `O(n)` | `O(n)` |
| 3. Hai lượt đếm | `O(n)` | `O(n)` |
| 4. Một lượt + xoá `(` từ cuối | `O(n)` | `O(n)` |
| 5. Stack, sửa list tại chỗ | `O(n)` | `O(n)` |

Kết quả bản thân đã cần `O(n)` bộ nhớ, nên `O(n)` là tối ưu.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [16]:
ALL = [SolutionStack, SolutionTwoPass, SolutionOnePassTrim, SolutionInPlace, Solution]


def check(s, with_brute=True):
    outs = [cls().minRemoveToMakeValid(s) for cls in ALL]
    for cls, o in zip(ALL, outs):
        assert is_correct(s, o), (cls.__name__, s, o)
    if with_brute:
        b = SolutionBrute().minRemoveToMakeValid(s)
        assert is_correct(s, b) and len(b) == len(outs[0])
    return outs[0]


for s, _ in examples:
    check(s)
for s in ["a", "(", ")", "()", ")(", "((()", "())", "abc", "(a(b)c)", "))((a", "a(b)c)d(e"]:
    print(repr(s), "->", repr(check(s)))

'a' -> 'a'
'(' -> ''
')' -> ''
'()' -> '()'
')(' -> ''
'((()' -> '()'
'())' -> '()'
'abc' -> 'abc'
'(a(b)c)' -> '(a(b)c)'
'))((a' -> 'a'
'a(b)c)d(e' -> 'a(b)cde'


### 12.2 Đáp án khác nhau nhưng đều đúng

Brute force và stack có thể ra chuỗi **khác nhau** — validator chấp nhận cả hai.

In [17]:
s = "(a)b)"
print("brute:", repr(SolutionBrute().minRemoveToMakeValid(s)), "| stack:", repr(Solution().minRemoveToMakeValid(s)))
print(is_correct(s, "(ab)"), is_correct(s, "(a)b"), is_correct(s, "ab"))
assert SolutionBrute().minRemoveToMakeValid(s) != Solution().minRemoveToMakeValid(s)

brute: '(ab)' | stack: '(a)b'
True True False


### 12.3 Ngẫu nhiên có seed + đối chiếu brute force

In [18]:
rng = random.Random(1249)
for _ in range(500):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("()()ab") for _ in range(n))
    check(s)
for _ in range(500):
    n = rng.randint(1, 300)
    s = "".join(rng.choice("((()))abc") for _ in range(n))
    check(s, with_brute=False)
print("1000 test ngẫu nhiên OK")

1000 test ngẫu nhiên OK


### 12.4 Test giới hạn `n = 10⁵`

In [19]:
big = ["".join(random.Random(7).choice("(()a)") for _ in range(10 ** 5)),
       "(" * 50000 + ")" * 50000,
       ")" * 50000 + "(" * 50000]
for s in big:
    t = time.perf_counter()
    res = check(s, with_brute=False)
    print(f"n={len(s)}: giữ lại {len(res)} ký tự, {(time.perf_counter() - t) * 1000:.0f} ms cho 5 cách + validator")

n=100000: giữ lại 0 ký tự, 74 ms cho 5 cách + validator
n=100000: giữ lại 100000 ký tự, 112 ms cho 5 cách + validator
n=100000: giữ lại 0 ký tự, 68 ms cho 5 cách + validator


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xử lý `)` thừa, quên `(` thừa

In [20]:
def wrong_only_close(s):
    out, bal = [], 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal == 0:
                continue
            bal -= 1
        out.append(c)
    return "".join(out)                 # SAI: chưa xoá '(' dư


res = wrong_only_close("a(b(c)d")
print("sai :", repr(res), is_valid(res), "| đúng:", repr(Solution().minRemoveToMakeValid("a(b(c)d")))

sai : 'a(b(c)d' False | đúng: 'ab(c)d'


**Vì sao sai:** `"a(b(c)d"` còn một `(` không bao giờ đóng → kết quả không hợp lệ. **Sửa:** thêm lượt 2 (phải → trái) hoặc xoá các chỉ số còn trong stack.

### Lỗi 2 — Xoá `(` thừa từ **bên trái**

In [21]:
def wrong_trim_left(s):
    first, bal = [], 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal == 0:
                continue
            bal -= 1
        first.append(c)
    i = 0
    while bal > 0:                      # SAI: xoá '(' trái nhất
        if first[i] == "(":
            first[i] = ""
            bal -= 1
        i += 1
    return "".join(first)


s = "(a)("
res = wrong_trim_left(s)
print("sai :", repr(res), is_valid(res), "| đúng:", repr(Solution().minRemoveToMakeValid(s)))

sai : 'a)(' False | đúng: '(a)'


**Vì sao sai:** `(` trái nhất ở `"(a)("` đang ghép với `)`; xoá nó để lại `"a)("` — không hợp lệ. `(` thừa thật sự là cái **cuối**. **Sửa:** xoá các `(` phải nhất (Cách 4) hoặc đúng các chỉ số còn trong stack.

### Lỗi 3 — Dùng `str.replace` để xoá ký tự

In [22]:
def wrong_replace(s):
    st, remove = [], []
    for i, c in enumerate(s):
        if c == "(":
            st.append(i)
        elif c == ")":
            if st:
                st.pop()
            else:
                remove.append(i)
    remove += st
    for i in sorted(remove, reverse=True):   # đi từ phải sang để chỉ số không bị lệch...
        s = s.replace(s[i], "", 1)           # ...nhưng SAI: replace xoá lần xuất hiện ĐẦU TIÊN
    return s


s = "(a)("
res = wrong_replace(s)
print("sai :", repr(res), is_valid(res), "| đúng:", repr(Solution().minRemoveToMakeValid(s)))

sai : 'a)(' False | đúng: '(a)'


**Vì sao sai:** cần xoá `(` ở vị trí 3, nhưng `s.replace('(', '', 1)` xoá `(` **đầu tiên** (vị trí 0, đang ghép cặp) → `"a)("` không hợp lệ. (Nếu duyệt chỉ số từ trái sang còn tệ hơn: sau mỗi lần xoá, các chỉ số phía sau bị dịch, có thể `IndexError`.) **Sửa:** đánh dấu theo chỉ số rồi dựng chuỗi mới một lần.

### Lỗi 4 — Lượt 2 quét trái → phải thay vì phải → trái

In [23]:
def wrong_second_pass_direction(s):
    first, bal = [], 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal == 0:
                continue
            bal -= 1
        first.append(c)
    out, extra = [], bal
    for c in first:                     # SAI: bỏ 'extra' dấu '(' đầu tiên gặp
        if c == "(" and extra > 0:
            extra -= 1
            continue
        out.append(c)
    return "".join(out)


s = "(a)(b"
res = wrong_second_pass_direction(s)
print("sai :", repr(res), is_valid(res), "| đúng:", repr(Solution().minRemoveToMakeValid(s)))

sai : 'a)(b' False | đúng: '(a)b'


**Vì sao sai:** giống Lỗi 2 — bỏ `(` trái nhất làm hỏng cặp `(a)`. **Sửa:** lượt 2 phải đi **từ phải sang trái** (đối xứng với lượt 1).

## 14. Mẹo & nhận dạng pattern

- Một loại ngoặc → **biến đếm** là đủ để kiểm tra; nhiều loại (`()[]{}`) → cần **stack ký tự**.
- Cần biết **ngoặc nào** sai → **stack chỉ số**.
- "Xoá ít nhất để hợp lệ, trả về một đáp án" → tham lam ghép gần nhất (`O(n)`); "trả về **mọi** đáp án" → BFS/backtracking (bài 301, Hard).
- Bài đối xứng trái/phải → **hai lượt quét ngược chiều**.
- Dựng chuỗi: list + `''.join`, tránh `+=` trong vòng lặp lớn và tránh `replace`.
- Bài cùng họ: 20 Valid Parentheses, 921 Minimum Add to Make Parentheses Valid, 1541 Minimum Insertions to Balance a Parentheses String, 301 Remove Invalid Parentheses, 32 Longest Valid Parentheses.

## 15. Biến thể

1. **Chỉ trả về số ngoặc cần xoá** → `min_removals(s)` (mục 3.5), `O(1)` bộ nhớ.
2. **921 — Thêm ít nhất để hợp lệ** → cùng số: `)` thừa + `(` thừa.
3. **Nhiều loại ngoặc** `()[]{}` → stack lưu `(ký tự, chỉ số)`; gặp đóng không khớp loại thì… (bài này trở nên khó hơn — tham lam không còn luôn tối ưu).
4. **301 — Trả về mọi chuỗi hợp lệ xoá ít nhất** → BFS theo số lần xoá, khử trùng bằng `set`.
5. **Có ký tự `*` đại diện `(`, `)` hoặc rỗng (678)** → đếm khoảng `[lo, hi]` của `bal`.

Demo biến thể 4 (cho chuỗi ngắn):

In [24]:
def all_min_valid(s):
    level = {s}
    while True:
        valid = {t for t in level if is_valid(t)}
        if valid:
            return sorted(valid)
        level = {t[:i] + t[i + 1:] for t in level for i in range(len(t)) if t[i] in "()"}


print(all_min_valid("lee(t(c)o)de)"))
print(all_min_valid("()())()"))

['lee(t(c)o)de', 'lee(t(c)ode)', 'lee(t(co)de)']
['(())()', '()()()']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Thử tập ngoặc bị xoá tăng dần | `O(2ᵖ · n)` | `O(n)` | Kiểm thử chuỗi ngắn |
| 2. Stack + set | Ghép bằng stack, xoá phần không ghép | `O(n)` | `O(n)` | Rõ ràng, dễ giải thích |
| 3. Hai lượt đếm | Trái→phải bỏ `)`, phải→trái bỏ `(` | `O(n)` | `O(n)` | Không muốn dùng stack |
| 4. Một lượt + cắt `(` cuối | Đếm `(` dư, xoá từ phải | `O(n)` | `O(n)` | Ngắn gọn, ít bộ nhớ phụ |
| 5. Stack, sửa list | Gán `''` tại chỗ | `O(n)` | `O(n)` | Code nộp gọn nhất |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minRemoveToMakeValid("(a(b)c")`?

**Bài 2 (tính tay):** `min_removals("))((a)")` bằng bao nhiêu?

**Bài 3 (điền TODO):** viết `min_add_to_make_valid(s)` (bài 921): số ngoặc **thêm** ít nhất để hợp lệ.

**Bài 4 (tự viết lại):** viết `min_remove_two_pass(s)` bằng hai lượt đếm (không nhìn Cách 3).

**Bài 5 (mini challenge):** viết `longest_valid_after_remove(s)` trả về độ dài chuỗi kết quả của bài (không dựng chuỗi), `O(1)` bộ nhớ phụ.

In [25]:
def min_add_to_make_valid(s):
    # TODO
    raise NotImplementedError


def min_remove_two_pass(s):
    # TODO
    raise NotImplementedError


def longest_valid_after_remove(s):
    # TODO: len(s) - min_removals(s), viết lại bằng một vòng lặp
    raise NotImplementedError


# Test gợi ý:
# assert min_add_to_make_valid("())") == 1 and min_add_to_make_valid("(((") == 3
# assert is_correct("a)b(c)d", min_remove_two_pass("a)b(c)d"))
# assert longest_valid_after_remove("lee(t(c)o)de)") == 12

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: chạy tay stack chỉ số.
- Mức 2: Bài 1: `(` ở vị trí 0 không bao giờ được đóng.
- Mức 3: Bài 2: hai `)` đầu thừa; hai `(` sau: một cái ghép với `)` cuối, một cái dư.

**Bài 3–4:**
- Mức 1: Bài 3 giống `min_removals`: mỗi ngoặc thừa cần thêm một ngoặc đối ứng.
- Mức 2: Bài 4: lượt 2 dùng `reversed(...)` và đổi vai trò `(`/`)`.
- Mức 3: nhớ đảo ngược kết quả lượt 2 trước khi `join`.

**Bài 5:**
- Mức 1: chỉ cần đếm, không dựng chuỗi.
- Mức 2: đếm `)` thừa và `(` dư như `min_removals`.
- Mức 3: `len(s) - extra_close - bal`.

## 19. Đáp án tham khảo

**Bài 1:** `"a(b)c"`. **Bài 2:** `3`.

In [26]:
assert Solution().minRemoveToMakeValid("(a(b)c") == "a(b)c"
assert min_removals("))((a)") == 3
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
def min_add_to_make_valid(s):
    bal = need = 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal == 0:
                need += 1
            else:
                bal -= 1
    return need + bal


def min_remove_two_pass(s):
    first, bal = [], 0
    for c in s:
        if c == ")" and bal == 0:
            continue
        if c == "(":
            bal += 1
        elif c == ")":
            bal -= 1
        first.append(c)
    out, bal = [], 0
    for c in reversed(first):
        if c == "(" and bal == 0:
            continue
        if c == ")":
            bal += 1
        elif c == "(":
            bal -= 1
        out.append(c)
    return "".join(reversed(out))


def longest_valid_after_remove(s):
    bal = extra = 0
    for c in s:
        if c == "(":
            bal += 1
        elif c == ")":
            if bal:
                bal -= 1
            else:
                extra += 1
    return len(s) - extra - bal


assert min_add_to_make_valid("())") == 1 and min_add_to_make_valid("(((") == 3
assert is_correct("a)b(c)d", min_remove_two_pass("a)b(c)d"))
assert longest_valid_after_remove("lee(t(c)o)de)") == 12
rng = random.Random(21)
for _ in range(300):
    s = "".join(rng.choice("(()a)") for _ in range(rng.randint(1, 40)))
    assert is_correct(s, min_remove_two_pass(s))
    assert longest_valid_after_remove(s) == len(Solution().minRemoveToMakeValid(s))
    assert min_add_to_make_valid(s) == min_removals(s)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (stack chỉ số):**

```python
class Solution:
    def minRemoveToMakeValid(self, s: str) -> str:
        chars, st = list(s), []
        for i, c in enumerate(chars):
            if c == "(":
                st.append(i)
            elif c == ")":
                if st: st.pop()
                else: chars[i] = ""
        for i in st:
            chars[i] = ""
        return "".join(chars)
```

**Checklist trước khi submit:**

- [ ] Xử lý **cả hai** loại thừa: `)` khi stack rỗng **và** `(` còn lại cuối cùng.
- [ ] Xoá theo **chỉ số**, không dùng `replace`.
- [ ] Nếu dùng đếm: lượt 2 đi **phải → trái**, hoặc xoá `(` **phải nhất**.
- [ ] Giữ nguyên mọi chữ cái và thứ tự.
- [ ] Dựng chuỗi bằng `''.join`.
- [ ] Test bằng validator (nhiều đáp án đúng).